In [134]:
from time import sleep

import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys

url = "https://chicagoengineersfoundation.awardspring.com/"

review_df = {'Reviewer'                  :  [],
             'Applicant'                 :  [],
             'Community Service / Work'  :  [],
             'Short Essay'               :  [],
             'Bonus/Discretionary Points':  [],
             'Notes'                     :  []
             }

review_feedback_df = pd.DataFrame(review_df)

driver = webdriver.Firefox()

In [135]:
review_feedback_df = pd.DataFrame(review_df)

In [136]:
driver.get(f'{url}')

In [137]:
# Login

email_xpath = '//*[@id="UserEmail"]'
email = 'cportiza221@gmail.com'

inputElement = driver.find_element(by=By.XPATH, value=email_xpath)
inputElement.send_keys(email)

pw_xpath = '//*[@id="Password"]'
pw = 'Cps45326551:))))'

inputElement = driver.find_element(by=By.XPATH, value=pw_xpath)
inputElement.send_keys(pw)

inputElement.send_keys(Keys.ENTER)

In [138]:
# Go to Reviewers
url = "https://chicagoengineersfoundation.awardspring.com/Admin/Users/Reviewers"

driver.get(f'{url}')

In [127]:
def get_reviewer_scores():
    commwork = 0
    essay = 0
    bonus = 0

    for score in range(0, 3):
        try: 
            xpath = f'//*[@id="score-value{score}"]'
            
            element = driver.find_element(by=By.XPATH, value=xpath)
        except Exception as e:
            print(f'Error occurred while finding score element {score}:{e} update xpath if necessary')
            continue

        try:
            copied_text = element.get_attribute('value')
        except Exception as e:
            print(f'Error occurred while copying text from score element {score}: {e}')
            continue
        
        if score == 0:
            commwork = int(copied_text) if copied_text else 0
        elif score == 1:
            essay = int(copied_text) if copied_text else 0
        elif score == 2:
            bonus = int(copied_text) if copied_text else 0

    try: 
        note_xpath = '/html/body/app-root/body/app-layout/admin-layout/div/section/div/main/div/reviewer/review-scoring-layout/div/div/div[2]/review-scoring-scores/div/div[2]/div/div/textarea'
        note_ele = driver.find_element(by=By.XPATH, value=note_xpath)
        notes = note_ele.get_attribute('value') or ''
    except Exception as e:
        print(f'Error occurred while finding/clicking notes element: {e}')
        return commwork, essay, bonus, ''

    return commwork, essay, bonus, notes


In [128]:
impersonate_user_xpath = '/html/body/section/div/main/div/div[1]/span[2]/input'
impersonate_user_xpath = '//*[@id="impersonate"]'


def get_reviewer_feedback(review_feedback_df, num_reviewers, start_pos, batch_number):
    sleep(2)
    reviewer_number = 1
    for x in range(start_pos, num_reviewers):
        url = "https://chicagoengineersfoundation.awardspring.com/Admin/Users/Reviewers"

        driver.get(f'{url}')
        sleep(2)

        # We need to move to a different page once we have finished the first page of reviewers
        if (batch_number == 2): 
            next_page_xpath = '/html/body/section/div/main/div/div[2]/ul/li[6]/a/span'
            inputElement = driver.find_element(by=By.XPATH, value=next_page_xpath)
            inputElement.click()
            sleep(2)

        re_xpath = f'/html/body/section/div/main/div/div[2]/table/tbody/tr[{x}]/td[1]/a'
        #print(re_xpath)
        try: 
            inputElement = driver.find_element(by=By.XPATH, value=re_xpath)
        except Exception as e: 
            print(f'Reviewer at position {x} is not assigned to any students')
            continue
        
        reviewer_name = inputElement.text
        inputElement.send_keys(Keys.ENTER)
        sleep(2)
        
        # Impersonate Reviewer
        try:
            inputElement = driver.find_element(by=By.XPATH, value=impersonate_user_xpath)
            inputElement.click()
            sleep(3)
        except Exception as e:
            sleep(4)
            inputElement = driver.find_element(by=By.XPATH, value=impersonate_user_xpath)
            inputElement.click()
            sleep(2)
        
        # Get the reviewer group 
        reviewer_group_path = f'/html/body/section/div/main/div/div/div/div/div/div/table/tbody/tr/td[1]/span/a'

        student_count_reviewed = 0
        student_count_not_reviewed = 0
        
        try:
            inputElement = driver.find_element(by=By.XPATH, value=reviewer_group_path)
        except Exception as e:
            print(f'Reviewer {reviewer_name} has no students to review')
            try:
                stop_impers_xpath = '//*[@id="stopImpersonate"]'
                stop_impers_element = driver.find_element(by=By.XPATH, value=stop_impers_xpath)
                stop_impers_element.click()
                sleep(3)
            except Exception as e:
                sleep(5)
                stop_impers_xpath = '//*[@id="stopImpersonate"]'
                stop_impers_element = driver.find_element(by=By.XPATH, value=stop_impers_xpath)
                stop_impers_element.click()
                sleep(2)
            continue
        
        inputElement = driver.find_element(by=By.XPATH, value=reviewer_group_path)
        inputElement.send_keys(Keys.ENTER)
        sleep(3)

        student_xpath = f'/html/body/app-root/body/app-layout/admin-layout/div/section/div/main/div/reviewer/review-group-applicants-layout/div/div[3]/table/tbody/tr[2]/td[1]'
        try:
            inputElement = driver.find_element(by=By.XPATH, value=student_xpath)
            inputElement.click()
            sleep(5)

            more_students = True
            while more_students:
                try: 
                    student_name_xpath = '/html/body/app-root/body/app-layout/admin-layout/div/section/div/main/div/reviewer/review-scoring-layout/div/div/div[2]/review-scoring-scores/div/div[2]/div/h4'
                    student_name = driver.find_element(by=By.XPATH, value=student_name_xpath).text
                except Exception as e:
                    print(f'Error occurred while finding student name: {e}, update xpath if necessary')

                commwork, essay, bonus, notes = get_reviewer_scores()

                if (commwork == 0 and essay == 0 and bonus == 0 and notes == ''):
                    print(f'Student {student_name} has not been reviewed or scored by reviewer {reviewer_name}')
                    student_count_not_reviewed += 1
                else: 
                    student_count_reviewed += 1
                    # print(student_name, commwork, essay, bonus, notes)
                    new_review_df = {'Reviewer'                 : reviewer_name,
                                    'Applicant'                 : student_name,
                                    'Community Service / Work'  : commwork,
                                    'Short Essay'               : essay,
                                    'Bonus/Discretionary Points': bonus,
                                    'Notes'                     : notes
                                    }
                    
                    review_feedback_df = pd.concat([review_feedback_df, pd.DataFrame.from_records([new_review_df])])

                try:
                    next_button_xpath = '/html/body/app-root/body/app-layout/admin-layout/div/section/div/main/div/reviewer/review-scoring-layout/div/div/div[1]/review-scoring-detail/div/div[2]/div[2]/div[1]/button[2]'
                    next_button_element = driver.find_element(by=By.XPATH, value=next_button_xpath)
                    next_button_element.send_keys(Keys.ENTER)
                    sleep(2)
                except Exception as e:
                    #print(f'No more students for reviewer {reviewer_name}')
                    more_students = False
                    
            sleep(2)
        except Exception as e:
            print(f'Error occurred while processing reviewer {reviewer_name}: {e}')

        sleep(2)

        # Stop Impersonating
        print(f"Reviewer {reviewer_number}: {reviewer_name} finished reviewing {student_count_reviewed}/{student_count_not_reviewed + student_count_reviewed} students")
        try:
            stop_impers_xpath = '//*[@id="stopImpersonate"]'
            stop_impers_element = driver.find_element(by=By.XPATH, value=stop_impers_xpath)
            stop_impers_element.click()
            sleep(2)
            reviewer_number += 1
        except Exception as e:
            sleep(5)
            stop_impers_xpath = '//*[@id="stopImpersonate"]'
            stop_impers_element = driver.find_element(by=By.XPATH, value=stop_impers_xpath)
            stop_impers_element.click()
            sleep(2)
    return review_feedback_df


In [129]:
review_feedback_df_1 = get_reviewer_feedback(review_feedback_df, 26, 1, 1)

Reviewer 1: Auduong, Vivian finished reviewing 13/13 students
Reviewer 2: Avram, Joe finished reviewing 13/13 students
Student Perez-Aguilera, Omar has not been reviewed or scored by reviewer B, Debbie
Student turner, seth has not been reviewed or scored by reviewer B, Debbie
Reviewer 3: B, Debbie finished reviewing 24/26 students
Reviewer Banks, Jr., Kevin has no students to review
Reviewer Bennett, Sid has no students to review
Reviewer Beuving, Henry has no students to review
Student Foster, Elijah has not been reviewed or scored by reviewer Birrell, Debbie
Student Hodrick, James has not been reviewed or scored by reviewer Birrell, Debbie
Student Richmond, Amir has not been reviewed or scored by reviewer Birrell, Debbie
Student Romero Orellana, Kevin has not been reviewed or scored by reviewer Birrell, Debbie
Reviewer 4: Birrell, Debbie finished reviewing 58/62 students
Reviewer Burns, Pat has no students to review
Reviewer Candiano, Charles has no students to review
Student Richmon

In [130]:
review_feedback_df_2 = get_reviewer_feedback(review_feedback_df, 26, 1, 2)

Reviewer 1: Juettner, Paul finished reviewing 17/17 students
Reviewer 2: Kallianis, McKenna finished reviewing 17/17 students
Reviewer Kennedy, Lisa has no students to review
Reviewer 3: Knobloch, Madison finished reviewing 17/17 students
Reviewer 4: Kuether, Jenny finished reviewing 16/16 students
Reviewer 5: Lev, Michael finished reviewing 18/18 students
Reviewer Levenfeld, Drew has no students to review
Reviewer Liston, John has no students to review
Reviewer Little, Nathan has no students to review
Reviewer Majeske, Jeana has no students to review
Reviewer Manning, Leo has no students to review
Reviewer McGann, Virginia has no students to review
Reviewer Quintero, Josie has no students to review
Reviewer Reviewer 1, College has no students to review
Reviewer Reviewer 2, College has no students to review
Reviewer Reviewer 3, College has no students to review
Reviewer Rienks, Steve has no students to review
Reviewer Riley, Tom has no students to review
Reviewer Rubert, Ellen has no s

In [131]:
#given that I have review_feedback_df_1 and review_feedback_df_2, I want to combine them into one dataframe
review_feedback_df = pd.concat([review_feedback_df_1, review_feedback_df_2], ignore_index=True)

In [132]:
review_feedback_df.drop_duplicates(inplace=True)
display(review_feedback_df.head(10))
review_feedback_df.to_excel('2026 CEF Reviewer Detailed Feedback.xlsx')
review_feedback_df = pd.read_excel('2026 CEF Reviewer Detailed Feedback.xlsx', engine='openpyxl')

,Reviewer,Applicant,Community Service / Work,Short Essay,Bonus/Discretionary Points,Notes
0,"Auduong, Vivian","Dorden, Joshua",15.0,45.0,10.0,
1,"Auduong, Vivian","Girolamo, Emmanuel",20.0,55.0,10.0,
2,"Auduong, Vivian","Pittman, Destiny",15.0,45.0,20.0,
3,"Auduong, Vivian","Richmond, Amir",10.0,30.0,10.0,
4,"Auduong, Vivian","Romero Orellana, Kevin",5.0,45.0,10.0,
5,"Auduong, Vivian","Ruiz, Christian",15.0,50.0,15.0,
7,"Auduong, Vivian","Scales, Eulanda",20.0,60.0,20.0,
8,"Auduong, Vivian","Sellis, Alex",15.0,50.0,20.0,
9,"Auduong, Vivian","Smith, Amajay",10.0,50.0,10.0,
10,"Auduong, Vivian","Szafranski, Jackson",20.0,55.0,15.0,


In [133]:
display(review_feedback_df.head(25))

,Unnamed: 0,Reviewer,Applicant,Community Service / Work,Short Essay,Bonus/Discretionary Points,Notes
0,0,"Auduong, Vivian","Dorden, Joshua",15,45,10,NaN
1,1,"Auduong, Vivian","Girolamo, Emmanuel",20,55,10,NaN
2,2,"Auduong, Vivian","Pittman, Destiny",15,45,20,NaN
3,3,"Auduong, Vivian","Richmond, Amir",10,30,10,NaN
4,4,"Auduong, Vivian","Romero Orellana, Kevin",5,45,10,NaN
5,5,"Auduong, Vivian","Ruiz, Christian",15,50,15,NaN
6,7,"Auduong, Vivian","Scales, Eulanda",20,60,20,NaN
7,8,"Auduong, Vivian","Sellis, Alex",15,50,20,NaN
8,9,"Auduong, Vivian","Smith, Amajay",10,50,10,NaN
9,10,"Auduong, Vivian","Szafranski, Jackson",20,55,15,NaN


In [ ]:
#review_feedback_df = get_reviewer_feedback(review_feedback_df, 17, 1)

In [ ]:
#review_feedback_df = get_reviewer_feedback(review_feedback_df, 11, 1)


In [ ]:
#review_feedback_df = pd.read_excel('2025 CEF Reviewer Detailed Feedback.xlsx')
#review_feedback_second_df = get_reviewer_feedback(review_feedback_df, 11, 1)

In [ ]:
# Merge review_feedback_df and review_feedback_second_df
#review_feedback_df = pd.concat([review_feedback_df, review_feedback_second_df])
#review_feedback_df.drop_duplicates(inplace=True)
#display(review_feedback_df.head(10))
#review_feedback_df.to_excel('2025 CEF Reviewer Detailed Feedback.xlsx')

In [ ]:
# Note: NEed to start on the Reviewers Screen
#review_feedback_df = get_reviewer_feedback(review_feedback_df, 17, 1)
#for next_num in range(1, 14):
#    next_rebutton_xpath = '/html/body/section/div/main/div/div[2]/div/ul/li[6]/a/span'
#    next_rebutton_element = driver.find_element(by=By.XPATH, value=next_rebutton_xpath)
#    next_rebutton_element.click()
#    sleep(2)
#    review_feedback_df = get_reviewer_feedback(review_feedback_df, next_num + 1, start_pos=next_num)

In [ ]:
#review_feedback_df.drop_duplicates(inplace=True)
#display(review_feedback_df.head(10))
#review_feedback_df.to_excel('2025 CEF Reviewer Detailed Feedback.xlsx')

In [ ]:
import googlemaps

In [ ]:
gmaps = googlemaps.Client(key='')

home_address = "3745 W Wilson, Chicago, IL 60625"
arrive_time = "2025-04-22T08:00:00-05:00"
gmaps.directions(home_address, "Walter Payton College Prep", mode="driving", arrival_time=arrive_time,region="us")